## Filter NACC data into cases with all correct answers, all incorrect answers and the ones in between

Created by: Sahana Kowshik

Date: 05/21/2025

In [1]:
import pandas as pd
import ast
import re
import json

In [2]:
# data_folder_name = "training_data_grpo"
data_folder_name = "training_data_grpo_modified_cog_np"

In [3]:
train_data = pd.read_csv(f"../../data/nacc/training_data/{data_folder_name}/train_summary.csv")
# data = pd.read_csv("../../adrd_simplified_evaluation/llm_answer_extractor/extracted_results_sub/Train/qwen25_3B_nacc_Train.csv")
data = pd.read_csv("../../adrd_simplified_evaluation/llm_answer_extractor/extracted_results_sub_modified_cog_np/Train/qwen25_3B_nacc_Train.csv")

In [4]:
data.head()

,problem,prompt,generated_text,finish_reason,ID,UNQ_ID,prediction,keys,group,extraction_type,extracted,ground_truth,ground_truth_text
0,"{'ID': 'NACC000034', 'patient_summary': '{""Sub...",NaN,Let's analyze the clinical details provided to...,stop,NACC000034,531642,C,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",valid,regex,NaN,C,Alzheimer's disease (AD)
1,"{'ID': 'NACC000034', 'patient_summary': '{""Sub...",NaN,To determine the main cause of this patient's ...,stop,NACC000034,531639,C,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",valid,regex,NaN,C,Alzheimer's disease (AD)
2,"{'ID': 'NACC000034', 'patient_summary': '{""Sub...",NaN,To determine the main cause of the patient's c...,stop,NACC000034,531638,C,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",valid,regex,NaN,C,Alzheimer's disease (AD)
3,"{'ID': 'NACC000034', 'patient_summary': '{""Sub...",NaN,To determine the main cause of the patient's c...,stop,NACC000034,531637,C,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",valid,regex,NaN,C,Alzheimer's disease (AD)
4,"{'ID': 'NACC000034', 'patient_summary': '{""Sub...",NaN,To determine the main cause of the patient's c...,stop,NACC000034,531636,C,"['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']",valid,regex,NaN,C,Alzheimer's disease (AD)


In [5]:
train_data.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary_prompt,visit_summary,COHORT
0,NACC814828,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","From the information available, determine the ...",A. Dementia (DE)\nB. Normal Cognition (NC),A,Dementia (DE),Cognitive status,<|im_start|>user\nYou will receive patient dat...,"The subject is an 83-year-old right-handed, En...",NACC
1,NACC142849,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Based on the clinical data, identify the neuro...",A. No listed option is correct\nB. Alzheimer's...,B,Alzheimer's disease pathology (AD) and Lewy bo...,Neuropath,<|im_start|>user\nYou will receive patient dat...,The subject is an 80-year-old male who present...,NACC
2,NACC698306,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","From the information available, determine the ...",A. Normal Cognition (NC)\nB. Dementia (DE),B,Dementia (DE),Cognitive status,<|im_start|>user\nYou will receive patient dat...,The patient is an 81-year-old White female who...,NACC
3,NACC517465,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Analyze the patient's information and identify...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,A,Mild Cognitive Impairment (MCI),Cognitive status,<|im_start|>user\nYou will receive patient dat...,The subject is a 61-year-old right-handed male...,NACC
4,NACC215680,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Using the information provided, identify the p...",A. Systemic and environmental factors includin...,B,Alzheimer's disease (AD),Primary etiology,<|im_start|>user\nYou will receive patient dat...,The subject is a 72-year-old male of Puerto Ri...,NACC


In [6]:
assert len(data['ID'].unique()) == len(train_data)
assert len(set(data['ID'].unique()).intersection(set(train_data['ID'].unique()))) == len(data['ID'].unique())

In [7]:
def pet_summary_with_invalid_keys(df):
    invalid = []
    sub_df = df[df['Q_TYPE'] == 'Amyloid PET'].reset_index(drop=True)
    for i, row in sub_df.iterrows():
        if "amyloid " in row['visit_summary'].lower() and "amyloid ang" not in row['visit_summary'].lower():
            invalid.append(row['ID'])
            
    return invalid

def dat_summary_with_invalid_keys(df):
    invalid = []
    sub_df = df[df['Q_TYPE'] == 'DATSCAN'].reset_index(drop=True)
    for i, row in sub_df.iterrows():
        if "datscan" in row['visit_summary'].lower():
            invalid.append(row['ID'])
            
    return invalid

In [8]:
invalid_ids_pet = pet_summary_with_invalid_keys(train_data)
invalid_ids_dat = dat_summary_with_invalid_keys(train_data)

invalid_df = train_data[(train_data['ID'].isin(invalid_ids_pet)) | (train_data['ID'].isin(invalid_ids_dat))]
len(invalid_df)

0

In [9]:
train_data = train_data[~train_data['ID'].isin(invalid_df['ID'])].reset_index(drop=True)

In [10]:
print(len(data['ID'].unique()))
print(len(train_data))

43064
43064


In [11]:
len(data[data['extraction_type'] == 'llm'])

59654

In [12]:
def modify(results_df):
    results_combined = results_df.copy()[['prediction','ground_truth', 'ID']]
    results_combined['prediction'] = results_combined['prediction']
    results_combined['correctness'] = results_combined['prediction'] == results_combined['ground_truth']
    results_combined = results_combined.reset_index(names=['problem']).reset_index(drop=True)
    return results_combined

In [13]:
data_trimmed = data.groupby('ID').head(8).reset_index(drop=True)

In [14]:
modified_data = modify(data)
# modified_data = modify(data_trimmed)

In [15]:
summary_df = modified_data.groupby('ID').agg(
    group_size=('correctness', 'size'),
    num_correct=('correctness', 'sum')  # True counts as 1
).reset_index()

In [16]:
group_size = summary_df['group_size'][0]
group_size

16

# Filter data

## Skywork

In [17]:
all_correct = summary_df[summary_df['num_correct'] == group_size].reset_index(drop=True)
all_incorrect = summary_df[(summary_df['num_correct'] == 0) & (summary_df['group_size'] == group_size)].reset_index(drop=True)
invalid_group = summary_df[summary_df['group_size'] != group_size].reset_index(drop=True)
remaining = summary_df[
    (~summary_df['ID'].isin(all_correct['ID'])) &
    (~summary_df['ID'].isin(all_incorrect['ID'])) &
    (~summary_df['ID'].isin(invalid_group['ID']))
].reset_index(drop=True)

In [18]:
remaining_data = train_data[train_data['ID'].isin(remaining['ID'])].reset_index(drop=True)
all_correct_data = train_data[train_data['ID'].isin(all_correct['ID'])].reset_index(drop=True)
all_incorrect_data = train_data[train_data['ID'].isin(all_incorrect['ID'])].reset_index(drop=True)

In [19]:
# print(f"Number of invalid cases: {len(invalid_data)}")
print(f"Number of all correct cases: {len(all_correct_data)}")
print(f"Number of all incorrect cases: {len(all_incorrect_data)}")
print(f"Number of selected cases: {len(remaining_data)}")

Number of all correct cases: 10993
Number of all incorrect cases: 2227
Number of selected cases: 29844


In [20]:
assert len(remaining_data) + len(all_correct_data) + len(all_incorrect_data) == len(train_data)

In [21]:
remaining_data.to_csv(f"../../data/nacc/training_data/{data_folder_name}/skywork_style/gs_{group_size}/train_filtered.csv", index=False)
all_correct_data.to_csv(f"../../data/nacc/training_data/{data_folder_name}/skywork_style/gs_{group_size}/all_correct.csv", index=False)
all_incorrect_data.to_csv(f"../../data/nacc/training_data/{data_folder_name}/skywork_style/gs_{group_size}/all_incorrect.csv", index=False)
# # invalid_df.to_csv(f"../../data/nacc/training_data/{data_folder_name}/skywork_style/invalid.csv", index=False)

## Stage wise

In [27]:
all_correct = summary_df[summary_df['num_correct'] == group_size].reset_index(drop=True)
stage_1 = summary_df[summary_df['num_correct'].isin(range(group_size // 4 * 3, group_size))].reset_index(drop=True)
stage_2 = summary_df[summary_df['num_correct'].isin(range(group_size // 4 * 2, group_size // 4 * 3))].reset_index(drop=True)
stage_3 = summary_df[summary_df['num_correct'].isin(range(group_size // 4 * 1, group_size // 4 * 2))].reset_index(drop=True)
stage_4 = summary_df[summary_df['num_correct'].isin(range(group_size // 4 * 0, group_size // 4 * 1))].reset_index(drop=True)


In [28]:
all_correct_data = train_data[train_data['ID'].isin(all_correct['ID'])].reset_index(drop=True)
stage_1_data = train_data[train_data['ID'].isin(stage_1['ID'])].reset_index(drop=True)
stage_2_data = train_data[train_data['ID'].isin(stage_2['ID'])].reset_index(drop=True)
stage_3_data = train_data[train_data['ID'].isin(stage_3['ID'])].reset_index(drop=True)
stage_4_data = train_data[train_data['ID'].isin(stage_4['ID'])].reset_index(drop=True)

In [29]:
stage_1_data["STAGE"] = 0
stage_2_data["STAGE"] = 1
stage_3_data["STAGE"] = 2
stage_4_data["STAGE"] = 3

In [30]:
# print(f"Number of invalid cases: {len(invalid_data)}")
print(f"Number of all correct cases: {len(all_correct_data)}")
print(f"Number of Stage 1 cases: {len(stage_1_data)}")
print(f"Number of Stage 2 cases: {len(stage_2_data)}")
print(f"Number of Stage 3 cases: {len(stage_3_data)}")
print(f"Number of Stage 4 cases: {len(stage_4_data)}")

Number of all correct cases: 10993
Number of Stage 1 cases: 13841
Number of Stage 2 cases: 6379
Number of Stage 3 cases: 5235
Number of Stage 4 cases: 6616


In [31]:
assert len(all_correct_data) + len(stage_1_data) + len(stage_2_data) + len(stage_3_data) + len(stage_4_data)  == len(train_data)

In [32]:
assert len(set(all_correct_data['ID']).intersection(set(stage_1_data['ID']))) == 0
assert len(set(stage_1_data['ID']).intersection(set(stage_2_data['ID']))) == 0
assert len(set(stage_2_data['ID']).intersection(set(stage_3_data['ID']))) == 0
assert len(set(stage_3_data['ID']).intersection(set(stage_4_data['ID']))) == 0
assert len(set(stage_4_data['ID']).intersection(set(all_correct_data['ID']))) == 0

In [33]:
stage_data = pd.concat([stage_1_data, stage_2_data, stage_3_data, stage_4_data]).reset_index(drop=True)

In [34]:
len(stage_data)

32071

In [35]:
stage_data.to_csv(f"../../data/nacc/training_data/{data_folder_name}/stage_wise/gs_{group_size}/stage_wise.csv", index=False)

In [ ]:
# all_correct_data.to_csv("../../data/nacc/training_data/training_data_grpo/stage_wise/all_correct.csv", index=False)
# stage_1_data.to_csv("../../data/nacc/training_data/training_data_grpo/stage_wise/stage_1.csv", index=False)
# stage_2_data.to_csv("../../data/nacc/training_data/training_data_grpo/stage_wise/stage_2.csv", index=False)
# stage_3_data.to_csv("../../data/nacc/training_data/training_data_grpo/stage_wise/stage_3.csv", index=False)
# stage_4_data.to_csv("../../data/nacc/training_data/training_data_grpo/stage_wise/stage_4.csv", index=False)

# Get a subset of each type of data for running experiments

In [ ]:
# train_data_sub = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
# train_data_sub.to_csv("../../data/nacc/training_data/training_data_grpo/subset/all_train_sub.csv", index=False)
# train_data_sub["Q_TYPE"].value_counts()

In [ ]:
remaining_data_sub = remaining_data.sample(n=4000, random_state=42).reset_index(drop=True)
remaining_data_sub.to_csv(f"../../data/nacc/training_data/training_data_grpo/subset/gs_{group_size}/train_filtered_sub.csv", index=False)
remaining_data_sub["Q_TYPE"].value_counts()

Q_TYPE
Cognitive status    1829
Primary etiology    1050
MCI subtype          672
Amyloid PET          190
Neuropath            155
Amyloid CSF           79
DATSCAN               25
Name: count, dtype: int64

In [61]:
stage_1_data_sub = stage_1_data.sample(n=1000, random_state=42)
stage_2_data_sub = stage_2_data.sample(n=1000, random_state=42)
stage_3_data_sub = stage_3_data.sample(n=1000, random_state=42)
stage_4_data_sub = stage_4_data.sample(n=1000, random_state=42)

In [62]:
combined_sub_df = pd.concat([stage_1_data_sub, stage_2_data_sub, stage_3_data_sub, stage_4_data_sub], axis=0).reset_index(drop=True)

In [ ]:
combined_sub_df.to_csv(f"../../data/nacc/training_data/training_data_grpo/subset/gs_{group_size}/stage_wise_sub.csv", index=False)
combined_sub_df["Q_TYPE"].value_counts()

Q_TYPE
Cognitive status    1706
Primary etiology    1135
MCI subtype          639
Neuropath            230
Amyloid PET          192
Amyloid CSF           82
DATSCAN               16
Name: count, dtype: int64

In [34]:
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [35]:
for key in pet_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "Amyloid PET"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [36]:
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]

In [37]:
for key in dat_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "DATSCAN"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break